# Раздел 7 · Квантование: экономия памяти в Qdrant

In [ ]:
%pip install qdrant-client sentence-transformers ipykernel --quiet


In [ ]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    ScalarQuantization, ScalarQuantizationConfig, ScalarType, QuantizationSearchParams, SearchParams
)

print("OK: библиотеки импортированы")


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [ ]:
CONTAINER_NAME = "qdrant_lecture"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())


In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("../books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


In [ ]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


## Квантование

Квантование настраивается на уже существующей коллекции через `update_collection` —
никакой новой коллекции и никакого пересчёта эмбеддингов не требуется: уже существующая
`BOOKS_COLLECTION` просто донастраивается.

### Пример — Scalar Quantization

In [ ]:
# update_collection меняет настройки уже существующей коллекции "на лету" - без пересоздания и без переупсерта точек
client.update_collection(
    collection_name=BOOKS_COLLECTION,
    quantization_config=ScalarQuantization(
        scalar=ScalarQuantizationConfig(type=ScalarType.INT8, quantile=0.99, always_ram=True)
    ),
)

info = client.get_collection(BOOKS_COLLECTION)
print("quantization_config:", info.config.quantization_config)


Реальный результат: `scalar=ScalarQuantizationConfig(type=<ScalarType.INT8: 'int8'>, quantile=0.99, always_ram=True, memory=None)`.
Обратите внимание, что итоговый конфиг квантования лежит на `info.config.quantization_config`,
а не внутри `info.config.params.vectors` — единственное отличие от конфига самих векторов,
легко перепутать при первой отладке.

## Задание 1

По запросу `"мрачная история об убийстве и мучительных муках совести"`
найдите топ-5 дважды: один раз **игнорируя** квантование (полные векторы, эталон), второй раз
**используя** квантованные векторы с честным rescore по полным. Сохраните результаты в `hits_full`
и `hits_quantized`.

**Используйте:** `client.query_points(...)`, `SearchParams(quantization=QuantizationSearchParams(ignore=True))` и
`SearchParams(quantization=QuantizationSearchParams(ignore=False, rescore=True))`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
hits_full = ...
hits_quantized = ...


In [ ]:
def check_ex1(hits_full, hits_quantized):
    ids_full = [h.id for h in hits_full]
    ids_quantized = [h.id for h in hits_quantized]
    assert len(ids_full) == 5 and len(ids_quantized) == 5, "ожидали top-5 в обоих случаях"
    overlap = len(set(ids_full) & set(ids_quantized))
    assert overlap >= 4, (
        "с rescore=True результат по квантованным векторам должен почти совпадать с полным, "
        "пересеклось только {} из 5".format(overlap)
    )
    print("OK: пересечение {}/5, ids_full={}, ids_quantized={}".format(overlap, ids_full, ids_quantized))
    print("ОТВЕТ ДЛЯ STEPIK:", overlap)


check_ex1(hits_full, hits_quantized)
